# BERTopic Parameter Sweep + Scaled min_cluster_size

For **cs.AI, cs.LG, cs.CV**

In [7]:
!pip install bertopic umap-learn hdbscan huggingface_hub pandas matplotlib -q


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from huggingface_hub import hf_hub_download
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

# ========================= CONFIG =========================
REPO_ID = "Sakhiur/empirical-rag-paradigm-benchmark"
REPO_TYPE = "dataset"
EMBEDDING_MODEL = "text-embedding-3-small"
CORPUS_TYPE = "abstracts"

TARGET_CATEGORIES = [
     "cs.IR", "cs.DB", "cs.SE","cs.CL", "cs.NE", "cs.DC", "cs.CR",       
]

# Original scaling logic (kept from your notebook)
MIN_CLUSTER_SIZE_FLOOR = 10
MIN_CLUSTER_SIZE_DIVISOR = 40   # you can change this here

def get_min_cluster_size(population_size: int) -> int:
    return max(MIN_CLUSTER_SIZE_FLOOR, population_size // MIN_CLUSTER_SIZE_DIVISOR)

# Fixed grid for sweeping
FIXED_MIN_CLUSTER_SIZES = [30, 50, 80, 120, 200]

PARAM_GRID = {
    "umap_n_neighbors": [10, 15, 30],
    "umap_n_components": [5, 10],
    "min_samples": [5, 10],
}

RANDOM_STATE = 42

SWEEP_DIR = Path(f"bertopic_sweep_{EMBEDDING_MODEL}")
SWEEP_DIR.mkdir(parents=True, exist_ok=True)
(SWEEP_DIR / "models").mkdir(exist_ok=True)
(SWEEP_DIR / "results").mkdir(exist_ok=True)

### Loaders (same as before)

In [9]:
def load_embeddings(category: str, corpus_type: str):
    repo_path = f"embeddings/{EMBEDDING_MODEL}/{corpus_type}/{category}.jsonl"
    local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    embeddings = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                embeddings[rec["chunk_id"]] = rec["embedding"]
    return embeddings


def load_source_records(category: str, corpus_type: str):
    if corpus_type == "abstracts":
        repo_path = f"abstracts/by_category/{category}.jsonl"
    else:
        repo_path = f"pdf_chunks/{category}.jsonl"
    local_path = hf_hub_download(repo_id=REPO_ID, repo_type=REPO_TYPE, filename=repo_path)
    records = {}
    with open(local_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                cid = rec["id"] if corpus_type == "abstracts" else f"{rec['paper_id']}_p{rec['page_num']}"
                records[cid] = rec
    return records


def get_display_text(record, corpus_type):
    return record["text"]

### Run Sweep (Fixed + Scaled)

In [10]:
results_summary = []

for category in TARGET_CATEGORIES:
    print(f"\n{'='*80}\nSWEEPING {category}\n{'='*80}")
    
    embeddings_dict = load_embeddings(category, CORPUS_TYPE)
    source_records = load_source_records(category, CORPUS_TYPE)
    
    chunk_ids = [cid for cid in embeddings_dict if cid in source_records]
    vectors = np.array([embeddings_dict[cid] for cid in chunk_ids], dtype=np.float32)
    texts = [get_display_text(source_records[cid], CORPUS_TYPE) for cid in chunk_ids]
    n_docs = len(chunk_ids)
    
    scaled_mcs = get_min_cluster_size(n_docs)
    print(f"Documents: {n_docs:,} | Scaled min_cluster_size = {scaled_mcs} (divisor={MIN_CLUSTER_SIZE_DIVISOR})")

    # 1. Test the original scaled value
    configs_to_try = [scaled_mcs] + FIXED_MIN_CLUSTER_SIZES
    configs_to_try = sorted(set(configs_to_try))  # remove duplicates

    for mcs in configs_to_try:
        for n_neigh in PARAM_GRID["umap_n_neighbors"]:
            for n_comp in PARAM_GRID["umap_n_components"]:
                for ms in PARAM_GRID["min_samples"]:
                    config_name = f"{category}_mcs{mcs}_nn{n_neigh}_nc{n_comp}_ms{ms}"
                    print(f"  → {config_name}", end=" ")

                    umap_model = UMAP(
                        n_neighbors=n_neigh,
                        n_components=n_comp,
                        min_dist=0.0,
                        metric="cosine",
                        random_state=RANDOM_STATE,
                        low_memory=True
                    )
                    hdbscan_model = HDBSCAN(
                        min_cluster_size=mcs,
                        min_samples=ms,
                        metric="euclidean",
                        cluster_selection_method="eom",
                        prediction_data=True,
                    )

                    topic_model = BERTopic(
                        umap_model=umap_model,
                        hdbscan_model=hdbscan_model,
                        calculate_probabilities=False,
                        verbose=False,
                    )

                    topics, _ = topic_model.fit_transform(texts, embeddings=vectors)

                    topic_info = topic_model.get_topic_info()
                    n_topics = len(topic_info[topic_info["Topic"] != -1])
                    n_outliers = int((np.array(topics) == -1).sum())
                    outlier_pct = 100 * n_outliers / n_docs if n_docs else 0

                    row = {
                        "category": category,
                        "n_docs": n_docs,
                        "min_cluster_size": mcs,
                        "is_scaled": mcs == scaled_mcs,
                        "umap_n_neighbors": n_neigh,
                        "umap_n_components": n_comp,
                        "min_samples": ms,
                        "n_topics": n_topics,
                        "outlier_pct": round(outlier_pct, 2),
                        "config_name": config_name,
                    }
                    results_summary.append(row)

                    if 15 <= n_topics <= 80 and outlier_pct < 45:
                        model_path = SWEEP_DIR / "models" / f"{config_name}.pkl"
                        topic_model.save(str(model_path))
                        print(f"✓ SAVED ({n_topics} topics, {outlier_pct:.1f}% outliers)")
                    else:
                        print(f"{n_topics} topics, {outlier_pct:.1f}% outliers")

# Save final results
df = pd.DataFrame(results_summary)
df.to_csv(SWEEP_DIR / "sweep_results.csv", index=False)

print("\n=== SWEEP FINISHED ===")
print(df.groupby("category")[["n_topics", "outlier_pct"]].describe().round(2))


SWEEPING cs.IR
Documents: 3,076 | Scaled min_cluster_size = 76 (divisor=40)
  → cs.IR_mcs30_nn10_nc5_ms5 

2026-07-17 10:42:09,106 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ SAVED (26 topics, 24.4% outliers)
  → cs.IR_mcs30_nn10_nc5_ms10 

2026-07-17 10:43:19,097 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ SAVED (23 topics, 33.6% outliers)
  → cs.IR_mcs30_nn10_nc10_ms5 

...

✓ SAVED (16 topics, 23.0% outliers)
  → cs.CR_mcs30_nn15_nc5_ms5 10 topics, 11.6% outliers
  → cs.CR_mcs30_nn15_nc5_ms10 

2026-07-17 13:51:22,962 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ SAVED (15 topics, 26.9% outliers)
  → cs.CR_mcs30_nn15_nc10_ms5 14 topics, 21.2% outliers
  → cs.CR_mcs30_nn15_nc10_ms10 

2026-07-17 13:52:05,058 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ SAVED (15 topics, 26.0% outliers)
  → cs.CR_mcs30_nn30_nc5_ms5 2 topics, 0.0% outliers
  → cs.CR_mcs30_nn30_nc5_ms10 2 topics, 0.0% outliers
  → cs.CR_mcs30_nn30_nc10_ms5 14 topics, 21.8% outliers
  → cs.CR_mcs30_nn30_nc10_ms10 2 topics, 0.0% outliers
  → cs.CR_mcs38_nn10_nc5_ms5 7 topics, 6.6% outliers
  → cs.CR_mcs38_nn10_nc5_ms10 12 topics, 26.3% outliers
  → cs.CR_mcs38_nn10_nc10_ms5 12 topics, 19.3% outliers
  → cs.CR_mcs38_nn10_nc10_ms10 7 topics, 7.3% outliers
  → cs.CR_mcs38_nn15_nc5_ms5 9 topics, 13.7% outliers
  → cs.CR_mcs38_nn15_nc5_ms10 10 topics, 23.7% outliers
  → cs.CR_mcs38_nn15_nc10_ms5 8 topics, 18.2% outliers
  → cs.CR_mcs38_nn15_nc10_ms10 4 topics, 2.2% outliers
  → cs.CR_mcs38_nn30_nc5_ms5 2 topics, 0.0% outliers
  → cs.CR_mcs38_nn30_nc5_ms10 2 topics, 0.0% outliers
  → cs.CR_mcs38_nn30_nc10_ms5 2 topics, 0.0% outliers
  → cs.CR_mcs38_nn30_nc10_ms10 2 topics, 0.0% outliers
  → cs.CR_mcs50_nn10_nc5_ms5 6 topics, 9.8% outliers
  → cs.CR_mcs50_nn10_nc5_ms10 6 topic

In [11]:
# Quick view of promising configs
promising = df[(df["n_topics"] >= 15) & (df["n_topics"] <= 80) & (df["outlier_pct"] < 45)]
print("\nPromising configurations:")
print(promising.sort_values(by=["category", "outlier_pct"]).head(15))


Promising configurations:
    category  n_docs  min_cluster_size  is_scaled  umap_n_neighbors  \
228    cs.CL   12418                50      False                10   
241    cs.CL   12418                80      False                10   
229    cs.CL   12418                50      False                10   
230    cs.CL   12418                50      False                10   
233    cs.CL   12418                50      False                15   
265    cs.CL   12418               200      False                10   
234    cs.CL   12418                50      False                15   
232    cs.CL   12418                50      False                15   
264    cs.CL   12418               200      False                10   
240    cs.CL   12418                80      False                10   
242    cs.CL   12418                80      False                10   
236    cs.CL   12418                50      False                30   
231    cs.CL   12418                50      False 